# mPFC Proteomics → Isocortex scRNA-seq Mapping

**Goal:** Score every cell in the Allen Mouse Brain Isocortex single-cell dataset by how highly it expresses a set of protein-encoding genes from a bulk mPFC neuronal membrane proteomics experiment, then determine which cortical cell types drive the signal.

## Scoring method
**`sc.tl.score_genes`** (Seurat AddModuleScore):  
`score(cell) = mean(query_gene_expr) − mean(expression-matched background gene expr)`

## Dataset
Allen Mouse Brain Cell Atlas — WMB-10X Isocortex (Yao et al. Nature 2023)  
Expression: log2(CPM/10 + 1) — pre-normalized in Allen h5ad files


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import anndata as ad
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
sc.settings.verbosity = 1

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'svg.fonttype': 'none',   # keep text as editable text in SVG, not paths
})

print(f'scanpy {sc.__version__} | anndata {ad.__version__}')

## 2. Download Allen Metadata (one-time, ~300 MB)

Downloads cell labels, UMAP coordinates, and cell type annotations for the Isocortex dataset. No expression data is downloaded here.

In [ ]:
from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache

# ─── SET THIS ────────────────────────────────────────────────────────────────
# Small cache — only metadata CSVs will be stored here, not expression files
METADATA_CACHE = Path('./allen_metadata_cache')
# ─────────────────────────────────────────────────────────────────────────────

METADATA_CACHE.mkdir(exist_ok=True)

abc_cache = AbcProjectCache.from_s3_cache(METADATA_CACHE)
abc_cache.load_manifest('releases/20230830/manifest.json')
print("Manifest loaded:", abc_cache.current_manifest)

In [ ]:
print('Loading cell metadata...')
cell_meta = abc_cache.get_metadata_dataframe(
    directory='WMB-10X',
    file_name='cell_metadata',
    dtype={'cell_label': str}
).set_index('cell_label')

# Keep only Isocortex cells — feature_matrix_label directly encodes the
# source expression file (e.g. 'WMB-10Xv2-Isocortex-1')
cell_meta = cell_meta[
    cell_meta['feature_matrix_label'].str.contains('Isocortex', case=False, na=False)
]
print(f'  Isocortex cells: {len(cell_meta):,}')
print(f'  Columns: {list(cell_meta.columns)}')


In [ ]:
# Cell type annotations live in a taxonomy directory, not WMB-10X.
# First, discover what directories and files are available in this manifest:
print("Available directories in manifest:")
for d in abc_cache.list_directories:
    print(f"  {d}")

ANNOT_COLS = ['class', 'subclass', 'supertype', 'neurotransmitter']
cluster_annot = None

# Try known taxonomy directory / filename combos across ABC Atlas release versions
for tax_dir in ['WMB-taxonomy', 'WMB-10X-taxonomy', 'allen_wmb_taxonomy', 'WMB-10X']:
    for fname in [
        'cluster_to_cluster_annotation_membership_pivoted',
        'cluster_annotation_term_set',
        'cluster_annotation_term',
        'cl.df_CCN202307220',
    ]:
        try:
            cluster_annot = abc_cache.get_metadata_dataframe(
                directory=tax_dir, file_name=fname
            )
            print(f"\n✓ Loaded cluster annotations: {tax_dir}/{fname}")
            print(f"  Columns: {list(cluster_annot.columns[:10])}")
            break
        except (KeyError, Exception):
            pass
    if cluster_annot is not None:
        break

if cluster_annot is None:
    # Print every metadata file in every directory so you can find the right one
    print("\nAuto-detection failed. Full metadata file listing:")
    for d in abc_cache.list_directories:
        try:
            files = abc_cache.list_metadata_files(d)
            if files:
                print(f"\n  [{d}]")
                for f in files:
                    print(f"    {f}")
        except Exception:
            pass
    raise RuntimeError(
        "Update tax_dir and fname in the loop above using the listing printed here."
    )

# Identify the cluster key column (index into cell_meta's cluster column)
cluster_key = next(
    (c for c in ['cluster_alias', 'cluster_label', 'cl', cluster_annot.columns[0]]
     if c in cluster_annot.columns),
    cluster_annot.columns[0]
)
cluster_annot = cluster_annot.set_index(cluster_key)

keep_cols = [c for c in ANNOT_COLS if c in cluster_annot.columns]
print(f"Annotation columns to attach: {keep_cols}")

join_key = next(c for c in ['cluster_alias', 'cluster_label'] if c in cell_meta.columns)
cell_meta = cell_meta.join(cluster_annot[keep_cols], on=join_key, how='left')

print("\nIsocortex cell class distribution:")
print(cell_meta['class'].value_counts().head(10).to_string())

In [ ]:
# UMAP coordinates — Isocortex 2D embedding
# First check: coordinates are sometimes already in cell_meta as 'x' / 'y'
umap_candidates = [c for c in cell_meta.columns if any(k in c.lower() for k in ['umap', ' x', '_x', ' y', '_y']) or c in ['x', 'y']]
print(f"UMAP-like columns already in cell_meta: {umap_candidates}")

if {'x', 'y'}.issubset(set(cell_meta.columns)):
    print("✓ UMAP coords already in cell_meta — no separate file needed.")
    umap_df = cell_meta[['x', 'y']]
    UMAP_X, UMAP_Y = 'x', 'y'
else:
    print("\nSearching for UMAP embedding file...")
    umap_df = None

    # Try known names across ABC Atlas releases
    for umap_dir in ['WMB-10X', 'WMB-taxonomy', 'WMB-10X-taxonomy']:
        for fname in [
            'umap_cell_embedding',
            'WMB_umap_cell_embedding',
            'cell_embedding',
            '2D_umap_embedding',
            'WMB-10X_umap',
        ]:
            try:
                umap_df = abc_cache.get_metadata_dataframe(directory=umap_dir, file_name=fname)
                print(f"✓ Loaded UMAP from {umap_dir}/{fname}")
                print(f"  Columns: {list(umap_df.columns)}")
                break
            except (KeyError, Exception):
                pass
        if umap_df is not None:
            break

    if umap_df is None:
        print("\nAuto-detection failed. Listing all metadata files to find the UMAP file:")
        for d in abc_cache.list_directories:
            try:
                files = abc_cache.list_metadata_files(d)
                if files:
                    print(f"\n  [{d}]")
                    for f in files:
                        print(f"    {f}")
            except Exception:
                pass
        raise RuntimeError(
            "Set umap_dir and fname in the loop above using the listing printed here."
        )

    if 'cell_label' in umap_df.columns:
        umap_df = umap_df.set_index('cell_label')

    print(f"\nUMAP file columns: {list(umap_df.columns)}")
    print("Set UMAP_X / UMAP_Y in the next cell to match the x and y column names shown above.")

In [ ]:
# If auto-detection found a separate UMAP file, set the coordinate column names here.
# (If coords were already in cell_meta the variables were set automatically above.)
if 'UMAP_X' not in dir() or 'UMAP_Y' not in dir():
    UMAP_X = 'x'    # ← adjust to match column names printed above
    UMAP_Y = 'y'    # ← adjust to match column names printed above

# Join UMAP coords into cell_meta (skip if already there)
if UMAP_X not in cell_meta.columns:
    cell_meta = cell_meta.join(umap_df[[UMAP_X, UMAP_Y]], how='left')

n_umap = cell_meta[[UMAP_X, UMAP_Y]].notna().all(axis=1).sum()
print(f"UMAP columns : {UMAP_X}, {UMAP_Y}")
print(f"Cells with UMAP coords: {n_umap:,} / {len(cell_meta):,}")

## 3. Define Your Proteomics Gene List

Mouse gene symbols are title-case: `Camk2a`, `Grin2b`, `Dlg4`, etc.

In [ ]:
gene_file = Path('~/PFCnrxn_hits.xlsx')

xl = pd.ExcelFile(gene_file)
print(f"Sheets: {xl.sheet_names}")

df = pd.read_excel(gene_file, sheet_name=0)
print(f"Columns: {list(df.columns)}")
print(df.head(3))

GENE_COL = 'gene_symbol'

proteomics_genes_raw = df[GENE_COL].dropna().str.strip().tolist()
print(f"\nInput gene list: {len(proteomics_genes_raw)} genes")
print(f"First 10: {proteomics_genes_raw[:10]}")

In [ ]:
# Output directory named after the proteomics input file
OUTPUT_DIR = Path(f'results_{gene_file.stem}')
OUTPUT_DIR.mkdir(exist_ok=True)

# Move any existing results files at the root into this directory
import shutil
_existing = [
    *Path('.').glob('proteomics_score_*.pdf'),
    *Path('.').glob('proteomics_score_*.png'),
    *Path('.').glob('score_by_*.csv'),
]
for _f in _existing:
    shutil.move(str(_f), OUTPUT_DIR / _f.name)
    print(f'  Moved {_f.name} → {OUTPUT_DIR}/')

print(f'Output directory: {OUTPUT_DIR}/')


## 4. Parse Allen Atlas Manifest for Isocortex Expression File URLs

We parse the local manifest to get the URLs of the Isocortex h5ad files on S3.  
Only Isocortex files will be selected and downloaded.

In [ ]:
import json, requests
from tqdm import tqdm

manifest_path = METADATA_CACHE / 'releases/20230830/manifest.json'
with open(manifest_path) as f:
    manifest = json.load(f)

# Extract all WMB-10X expression file entries (log2-normalized, already CPM-scaled)
expr_records = []
for dataset in ['WMB-10Xv2', 'WMB-10Xv3']:
    for matrix_name, matrix_info in (
        manifest.get('file_listing', {})
                .get(dataset, {})
                .get('expression_matrices', {})
                .items()
    ):
        for norm in ['log2', 'raw']:
            entry = matrix_info.get(norm, {}).get('files', {}).get('h5ad', {})
            if entry:
                expr_records.append({
                    'key':      f"{dataset}/{matrix_name}/{norm}",
                    'dataset':  dataset,
                    'region':   matrix_name,
                    'norm':     norm,
                    'url':      entry['url'],
                    'size_gb':  entry.get('size', 0) / 1e9,
                })

expr_df = pd.DataFrame(expr_records)
log2_files = expr_df[expr_df['norm'] == 'log2'].sort_values(['dataset', 'region'])

# Show only Isocortex-relevant files for clarity
iso_preview = log2_files[log2_files['region'].str.contains('Isocortex', case=False, na=False)]
print(f"All region files in manifest: {len(log2_files)}  (Isocortex shown below)")
print(iso_preview[['dataset', 'region', 'size_gb']].to_string(index=False))

In [ ]:
# Expression files to download — Isocortex only
REGION_FILTER = ['Isocortex']

mask = log2_files['region'].apply(
    lambda r: any(filt.lower() in r.lower() for filt in REGION_FILTER)
)
selected_files = log2_files[mask]

print(f'Files selected : {len(selected_files)}')
print(f'Total download : ~{selected_files["size_gb"].sum():.1f} GB (downloaded once, kept as cache)')
print()
print(selected_files[['dataset', 'region', 'size_gb']].to_string(index=False))


## 5. Validate Gene List and Build Background Pool

In [ ]:
# Load WMB-10X gene metadata (already downloaded, small file)
gene_meta = abc_cache.get_metadata_dataframe('WMB-10X', 'gene')
print(f"WMB gene metadata columns: {list(gene_meta.columns)}")

# Find gene symbol column
GENE_SYM_COL = next(
    (c for c in ['gene_symbol', 'symbol', 'gene_name', 'feature_name']
     if c in gene_meta.columns),
    gene_meta.columns[0]
)
print(f"Using gene symbol column: '{GENE_SYM_COL}'")

dataset_genes  = set(gene_meta[GENE_SYM_COL].tolist())
lower_to_atlas = {g.lower(): g for g in dataset_genes}

query_genes, genes_missing = [], []
for g in proteomics_genes_raw:
    if g in dataset_genes:
        query_genes.append(g)
    elif g.lower() in lower_to_atlas:
        corrected = lower_to_atlas[g.lower()]
        query_genes.append(corrected)
        print(f"  Case-corrected: {g} → {corrected}")
    else:
        genes_missing.append(g)

query_genes = list(dict.fromkeys(query_genes))
print(f"\n✓ Matched: {len(query_genes)} / {len(proteomics_genes_raw)}")
print(f"✗ Missing: {len(genes_missing)}")
if genes_missing:
    print(f"  {genes_missing[:20]}")

In [ ]:
# Background gene pool for score_genes control sampling
np.random.seed(42)
n_background = max(500, 10 * len(query_genes))
n_background = min(n_background, 3000)

other_genes    = [g for g in gene_meta[GENE_SYM_COL].tolist() if g not in set(query_genes)]
background_genes = list(np.random.choice(other_genes, size=min(n_background, len(other_genes)), replace=False))
genes_to_load  = set(query_genes) | set(background_genes)

print(f"Query genes      : {len(query_genes)}")
print(f"Background genes : {len(background_genes)}")
print(f"Total to extract : {len(genes_to_load)} unique genes per section file")

## 6. Download Allen Expression Files (Permanent Cache)

Files are downloaded once to `expression_cache/` and kept permanently.
On re-run, already-present files are detected and skipped — no re-download needed.
Each file is ~2–5 GB; total size for the selected regions is shown above.

In [ ]:
EXPR_CACHE = Path('./expression_cache')   # permanent local cache for Allen h5ad files
EXPR_CACHE.mkdir(exist_ok=True)

def download_file(url, dest_path):
    """Download url → dest_path with progress bar. No-op if file already exists."""
    if dest_path.exists():
        print(f'  ✓ cached  {dest_path.name}')
        return
    r = requests.get(url, stream=True, timeout=60)
    r.raise_for_status()
    total = int(r.headers.get('content-length', 0))
    with open(dest_path, 'wb') as f, \
         tqdm(total=total, unit='B', unit_scale=True, desc=dest_path.name, leave=False) as bar:
        for chunk in r.iter_content(chunk_size=1 << 20):
            f.write(chunk)
            bar.update(len(chunk))

# ── Phase 1: download any missing files ──────────────────────────────────────
print('Checking / downloading expression files...')
file_paths = []
for _, row in tqdm(selected_files.iterrows(), total=len(selected_files), desc='Files'):
    fname     = row['url'].split('/')[-1]
    dest_path = EXPR_CACHE / fname
    download_file(row['url'], dest_path)
    file_paths.append(dest_path)

# ── Phase 2: load target genes from cached files ─────────────────────────────
# var_names in Allen h5ad files are Ensembl IDs; gene symbols are in var['gene_symbol'].
# Map genes_to_load (symbols) → Ensembl IDs, subset, then rename back to symbols.
print('\nLoading target genes from cached files...')
adatas = []
for dest_path in tqdm(file_paths, desc='Loading'):
    chunk = ad.read_h5ad(dest_path)
    ensembl_to_sym = chunk.var['gene_symbol'].to_dict()           # {ENSMUSG…: symbol}
    sym_to_ensembl = {v: k for k, v in ensembl_to_sym.items()}   # {symbol: ENSMUSG…}
    target_ensembl = [sym_to_ensembl[g] for g in genes_to_load if g in sym_to_ensembl]
    if target_ensembl:
        sub = chunk[:, target_ensembl].copy()
        sub.var_names = pd.Index([ensembl_to_sym[e] for e in sub.var_names])
        sub.var_names.name = 'gene_symbol'
        adatas.append(sub)
    del chunk

if not adatas:
    raise RuntimeError('No genes found in any cached file. Check gene names.')

print('Concatenating...')
adata = ad.concat(adatas, axis=0, merge='same')
del adatas

print(f'\nFinal AnnData: {adata.shape[0]:,} cells x {adata.shape[1]} genes')
print(adata)


## 7. Attach Allen Metadata (UMAP + Cell Type Labels)

In [ ]:
# The h5ad obs_names are Allen cell_labels — join directly with cell_meta
adata.obs_names.name = 'cell_label'

overlap = len(adata.obs_names.intersection(cell_meta.index))
print(f"Cells with matching Allen cell_label: {overlap:,} / {len(adata):,}")

if overlap == 0:
    print("\nSample adata obs_names :", list(adata.obs_names[:5]))
    print("Sample cell_meta index  :", list(cell_meta.index[:5]))
    raise RuntimeError("No overlap — inspect sample values above for format mismatch.")

# Attach UMAP coordinates and cell type labels
adata.obs = adata.obs.join(
    cell_meta[[UMAP_X, UMAP_Y] + keep_cols + ['region_of_interest_acronym']],
    how='left'
)

n_umap  = adata.obs[[UMAP_X, UMAP_Y]].notna().all(axis=1).sum()
n_class = adata.obs['class'].notna().sum()
print(f"Cells with UMAP coords : {n_umap:,}")
print(f"Cells with class label : {n_class:,}")

## 8. Score Each Cell (AddModuleScore)

In [ ]:
# Identify which query genes are actually present in the loaded expression data
query_in_adata = [g for g in query_genes if g in adata.var_names]
print(f'Query genes in adata : {len(query_in_adata)} / {len(query_genes)}')
if len(query_in_adata) == 0:
    raise RuntimeError('No query genes in adata — check gene name format.')
print(f'Example genes: {query_in_adata[:5]}')

In [ ]:
# Score each cell — Seurat AddModuleScore equivalent
# ctrl_size and n_bins scale with background pool to ensure enough control genes per bin
n_bg = len(background_genes)
n_bins = min(25, max(5, n_bg // 40))
ctrl_size = max(10, min(50, n_bg // n_bins))

print(f"score_genes settings: n_bins={n_bins}, ctrl_size={ctrl_size}")

sc.tl.score_genes(
    adata,
    gene_list=query_in_adata,
    ctrl_size=ctrl_size,
    n_bins=n_bins,
    score_name='proteomics_score',
    random_state=42,
    use_raw=False
)

print("\nScore statistics:")
print(adata.obs['proteomics_score'].describe().round(4))

In [ ]:
# Score distribution
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(adata.obs['proteomics_score'], bins=150, color='steelblue', alpha=0.85, edgecolor='none')
ax.axvline(0, color='crimson', linewidth=1.2, linestyle='--', label='score = 0')
ax.set_xlabel('Proteomics Gene Score')
ax.set_ylabel('Number of cells')
ax.set_title(f'Per-cell proteomics score  (n={len(adata):,} cells, {len(query_in_adata)} genes)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Attach scores to cell_meta (already filtered to Isocortex)
cell_meta_scored = cell_meta.join(
    adata.obs[['proteomics_score']],
    how='left'
)
n_scored = cell_meta_scored['proteomics_score'].notna().sum()
print(f'Cells with scores: {n_scored:,} / {len(cell_meta_scored):,}')


In [ ]:
# The Allen Atlas log2 files are already normalized (log2(CPM/10 + 1)).
# No further normalization needed — just confirm and proceed.
import scipy.sparse as sp
X_sample = adata.X[:5, :3]
if sp.issparse(X_sample):
    X_sample = X_sample.toarray()
print("Expression sample (log2-normalized, first 5 cells × 3 genes):")
print(X_sample)
print(f"\nValue range — min: {adata.X.min():.3f}, max: {adata.X.max():.3f}")

## 9. Visualize Scores on Isocortex UMAP

In [ ]:
SCORE_COL = 'proteomics_score'

has_score = cell_meta_scored[SCORE_COL].notna()
has_umap  = cell_meta_scored[[UMAP_X, UMAP_Y]].notna().all(axis=1)

scored   = cell_meta_scored[has_score & has_umap]
unscored = cell_meta_scored[~has_score & has_umap]  # Isocortex cells not in loaded files

rng = np.random.default_rng(0)
bg_idx  = rng.choice(len(unscored), size=min(200_000, len(unscored)), replace=False)
bg_plot = unscored.iloc[bg_idx]

vmin = scored[SCORE_COL].quantile(0.02)
vmax = scored[SCORE_COL].quantile(0.98)

print(f'Scored cells  : {len(scored):,}')
print(f'BG dots shown : {len(bg_plot):,}')
print(f'Color range   : [{vmin:.3f}, {vmax:.3f}]')


In [ ]:
# ── Cell-type label mapping: Isocortex subclasses only ──────────────────────
SUBCLASS_TO_LABEL = {
    # Cortical excitatory layers (superficial → deep)
    '007 L2/3 IT CTX Glut':     'L2/3 IT',
    '006 L4/5 IT CTX Glut':     'L4/5 IT',
    '005 L5 IT CTX Glut':       'L5 IT',
    '022 L5 ET CTX Glut':       'L5 ET',
    '032 L5 NP CTX Glut':       'L5 NP',
    '030 L6 CT CTX Glut':       'L6 CT',
    '004 L6 IT CTX Glut':       'L6 IT',
    '029 L6b CTX Glut':         'L6b',
    # Cortical inhibitory subtypes
    '052 Pvalb Gaba':            'Pvalb',
    '051 Pvalb chandelier Gaba': 'Pvalb',
    '053 Sst Gaba':              'Sst',
    '056 Sst Chodl Gaba':        'Sst',
    '046 Vip Gaba':              'Vip',
    '049 Lamp5 Gaba':            'Lamp5',
    '050 Lamp5 Lhx6 Gaba':      'Lamp5',
    '047 Sncg Gaba':             'Sncg',
    # Non-neuronal (cortical)
    '317 Astro-CB NN':           'Astro',
    '318 Astro-NT NN':           'Astro',
    '319 Astro-TE NN':           'Astro',
    '320 Astro-OLF NN':          'Astro',
    '321 Astroependymal NN':     'Astro',
    '326 OPC NN':                'OPC',
    '327 Oligo NN':              'Oligo',
    '334 Microglia NN':          'Microglia',
    '333 Endo NN':               'Vascular',
    '330 VLMC NN':               'Vascular',
    '331 Peri NN':               'Vascular',
    '332 SMC NN':                'Vascular',
}

LABEL_COLORS = {
    # Cortical excitatory — red → brown (superficial → deep)
    'L2/3 IT':   '#e63946',
    'L4/5 IT':   '#f4811f',
    'L5 IT':     '#f9a825',
    'L5 ET':     '#c77800',
    'L5 NP':     '#a0522d',
    'L6 CT':     '#7b3410',
    'L6 IT':     '#cd853f',
    'L6b':       '#deb887',
    # Cortical inhibitory — purple/blue
    'Pvalb':     '#5b4fcf',
    'Sst':       '#9c27b0',
    'Vip':       '#1976d2',
    'Lamp5':     '#00acc1',
    'Sncg':      '#26a69a',
    # Non-neuronal — muted grays
    'Astro':     '#9e9e9e',
    'Oligo':     '#546e7a',
    'OPC':       '#b0bec5',
    'Microglia': '#78909c',
    'Vascular':  '#cfd8dc',
    'Other':     '#ececec',
}

# Draw order: Other at bottom, non-neuronal next, neurons on top
PLOT_ORDER = [
    'Other', 'Vascular', 'Microglia', 'OPC', 'Oligo', 'Astro',
    'Sncg', 'Lamp5', 'Vip', 'Sst', 'Pvalb',
    'L6b', 'L6 IT', 'L6 CT', 'L5 NP', 'L5 ET', 'L5 IT', 'L4/5 IT', 'L2/3 IT',
]

if 'subclass' in scored.columns:
    scored_labeled = scored.copy()
    scored_labeled['cell_type_label'] = (
        scored_labeled['subclass'].map(SUBCLASS_TO_LABEL).fillna('Other')
    )
else:
    scored_labeled = scored.copy()
    scored_labeled['cell_type_label'] = 'Other'
    print("Warning: 'subclass' column not found — all cells labeled 'Other'")

PLOT_ORDER = [l for l in PLOT_ORDER if l in scored_labeled['cell_type_label'].values]

print('Label counts:')
print(scored_labeled['cell_type_label'].value_counts().to_string())


In [ ]:
import matplotlib.patheffects as pe

# ── Figure 1: Cell type labels ────────────────────────────────────────────
fig1, ax = plt.subplots(figsize=(11, 9))
fig1.patch.set_facecolor('white')

ax.scatter(bg_plot[UMAP_X], bg_plot[UMAP_Y],
           c='#d8d8d8', s=1, alpha=0.25, rasterized=True, linewidths=0)

for label in PLOT_ORDER:
    m = scored_labeled['cell_type_label'] == label
    if m.sum() == 0:
        continue
    ax.scatter(
        scored_labeled.loc[m, UMAP_X],
        scored_labeled.loc[m, UMAP_Y],
        c=[LABEL_COLORS.get(label, '#aaaaaa')],
        s=1,
        alpha=0.18 if label == 'Other' else 0.5,
        rasterized=True, linewidths=0,
        label=f"{label}  ({m.sum():,})",
    )

for label in PLOT_ORDER:
    if label == 'Other':
        continue
    m = scored_labeled['cell_type_label'] == label
    if m.sum() < 200:
        continue
    cx = scored_labeled.loc[m, UMAP_X].median()
    cy = scored_labeled.loc[m, UMAP_Y].median()
    ax.text(
        cx, cy, label,
        fontsize=7, ha='center', va='center', fontweight='bold',
        color='black', zorder=10,
        path_effects=[pe.withStroke(linewidth=2.5, foreground='white')],
    )

ax.set_title('Isocortex — Cell Type Labels', fontsize=13, pad=10)
ax.axis('off')
leg = ax.legend(
    markerscale=10, fontsize=7, loc='lower left', ncol=2,
    frameon=True, framealpha=0.9, bbox_to_anchor=(0.0, 0.0),
    title='Cell type  (n cells in scored set)', title_fontsize=7,
)
for lh in leg.legend_handles:
    lh.set_alpha(1.0)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'umap_celltypes.pdf', dpi=600, bbox_inches='tight')
plt.savefig(OUTPUT_DIR / 'umap_celltypes.svg', dpi=600, bbox_inches='tight')
plt.savefig(OUTPUT_DIR / 'umap_celltypes.png', dpi=600, bbox_inches='tight')
plt.show()
print(f"Saved {OUTPUT_DIR}/umap_celltypes.*")

# ── Figure 2: Proteomics score overlay ───────────────────────────────────
fig2, ax = plt.subplots(figsize=(10, 9))
fig2.patch.set_facecolor('white')

ax.scatter(bg_plot[UMAP_X], bg_plot[UMAP_Y],
           c='#d8d8d8', s=1, alpha=0.3, rasterized=True, linewidths=0)

scored_sorted = scored.sort_values(SCORE_COL)
sc_plot = ax.scatter(
    scored_sorted[UMAP_X], scored_sorted[UMAP_Y],
    c=scored_sorted[SCORE_COL],
    cmap='RdYlBu_r', vmin=vmin, vmax=vmax,
    s=1, alpha=0.6, rasterized=True, linewidths=0,
)
cbar = plt.colorbar(sc_plot, ax=ax, shrink=0.55, pad=0.02)
cbar.set_label('Proteomics Score (AddModuleScore)', fontsize=10)
cbar.ax.tick_params(labelsize=8)

ax.set_title('mPFC Neuronal Membrane Proteomics Score', fontsize=13, pad=10)
ax.axis('off')

plt.suptitle(
    f'{len(query_in_adata)} proteomics genes  |  {len(scored):,} cells  |  '
    f'Allen Isocortex  (Yao et al. 2023)',
    fontsize=10, y=1.01, color='gray',
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'umap_score.pdf', dpi=600, bbox_inches='tight')
plt.savefig(OUTPUT_DIR / 'umap_score.svg', dpi=600, bbox_inches='tight')
plt.savefig(OUTPUT_DIR / 'umap_score.png', dpi=600, bbox_inches='tight')
plt.show()
print(f"Saved {OUTPUT_DIR}/umap_score.*")


In [ ]:
p75 = scored[SCORE_COL].quantile(0.75)
high = scored[scored[SCORE_COL] > p75]

fig, ax = plt.subplots(figsize=(10, 9))
ax.scatter(bg_plot[UMAP_X], bg_plot[UMAP_Y],
           c='#ebebeb', s=0.03, alpha=0.4, rasterized=True, linewidths=0)
sc2 = ax.scatter(high[UMAP_X], high[UMAP_Y],
                 c=high[SCORE_COL], cmap='plasma',
                 vmin=p75, vmax=vmax,
                 s=0.1, alpha=0.7, rasterized=True, linewidths=0)
plt.colorbar(sc2, ax=ax, shrink=0.5, label=f'Score (top 25%, >{p75:.3f})')
ax.set_title(f'Isocortex cells — top 25% proteomics score  (n={len(high):,})', fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'proteomics_score_top25pct.pdf', dpi=600, bbox_inches='tight')
plt.savefig(OUTPUT_DIR / 'proteomics_score_top25pct.svg', dpi=600, bbox_inches='tight')
plt.show()


## 10. Cell Type Enrichment — Isocortex Subclasses

Scores broken down by the major cortical subclasses: excitatory layers, inhibitory subtypes, and non-neuronal populations.

In [ ]:
# Use the clean labels from cell 30 (SUBCLASS_TO_LABEL / scored_labeled)
# Drop 'Other' — analyze only identified cortical/non-neuronal types
NEURONAL_LABELS    = {'L2/3 IT','L4/5 IT','L5 IT','L5 ET','L5 NP',
                      'L6 CT','L6 IT','L6b','Pvalb','Sst','Vip','Lamp5','Sncg'}
NONNEURONAL_LABELS = {'Astro','Oligo','OPC','Microglia','Vascular'}

labeled_scored = scored_labeled[scored_labeled['cell_type_label'] != 'Other'].copy()
labeled_scored['is_neuronal'] = labeled_scored['cell_type_label'].isin(NEURONAL_LABELS)

VIOLIN_ORDER = [
    'L2/3 IT','L4/5 IT','L5 IT','L5 ET','L5 NP','L6 CT','L6 IT','L6b',
    'Pvalb','Sst','Vip','Lamp5','Sncg',
    'Astro','Oligo','OPC','Microglia','Vascular',
]
VIOLIN_ORDER = [l for l in VIOLIN_ORDER if l in labeled_scored['cell_type_label'].values]

subtype_stats = (
    labeled_scored.groupby('cell_type_label')[SCORE_COL]
    .agg(median='median', mean='mean', n='count', std='std')
    .reindex(VIOLIN_ORDER)
    .dropna()
)

print('=== SUBTYPE SCORE STATS ===')
print(subtype_stats.round(4).to_string())


In [ ]:
# Sample up to 5000 cells per type to keep plot manageable
violin_data = (
    labeled_scored[labeled_scored['cell_type_label'].isin(VIOLIN_ORDER)]
    .groupby('cell_type_label', group_keys=False)
    .apply(lambda g: g.sample(min(len(g), 5000), random_state=0))
)

n_neuron  = sum(1 for l in VIOLIN_ORDER if l in NEURONAL_LABELS)
palette   = {l: LABEL_COLORS.get(l, '#aaaaaa') for l in VIOLIN_ORDER}

fig, ax = plt.subplots(figsize=(17, 5.5))
sns.violinplot(
    data=violin_data, x='cell_type_label', y=SCORE_COL,
    order=VIOLIN_ORDER, palette=palette,
    cut=0, bw=0.2, linewidth=0.8, ax=ax,
)
# Divider between neuronal and non-neuronal
ax.axvline(n_neuron - 0.5, color='#444', lw=1.5, ls='--', alpha=0.55)
ax.axhline(0, color='crimson', lw=1, ls='--', alpha=0.6)

ymax = ax.get_ylim()[1]
ax.text(n_neuron / 2 - 0.5, ymax * 0.97, 'Neuronal',
        ha='center', fontsize=9, color='#444', style='italic')
ax.text(n_neuron + (len(VIOLIN_ORDER) - n_neuron) / 2 - 0.5, ymax * 0.97, 'Non-neuronal',
        ha='center', fontsize=9, color='#444', style='italic')

ax.set_xlabel('')
ax.set_ylabel('Proteomics Score', fontsize=11)
ax.set_title('mPFC Proteomics Score by Cortical Cell Subtype — Isocortex', fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'score_violin_subtypes.pdf', dpi=600, bbox_inches='tight')
plt.savefig(OUTPUT_DIR / 'score_violin_subtypes.svg', dpi=600, bbox_inches='tight')
plt.savefig(OUTPUT_DIR / 'score_violin_subtypes.png', dpi=600, bbox_inches='tight')
plt.show()
print(f'Saved {OUTPUT_DIR}/score_violin_subtypes.pdf')


In [ ]:
from scipy import stats as scipy_stats

neuro_scores    = labeled_scored[labeled_scored['is_neuronal']][SCORE_COL]
nonneuro_scores = labeled_scored[~labeled_scored['is_neuronal']][SCORE_COL]
excit_labels    = {'L2/3 IT','L4/5 IT','L5 IT','L5 ET','L5 NP','L6 CT','L6 IT','L6b'}
inhib_labels    = {'Pvalb','Sst','Vip','Lamp5','Sncg'}
excit_scores    = labeled_scored[labeled_scored['cell_type_label'].isin(excit_labels)][SCORE_COL]
inhib_scores    = labeled_scored[labeled_scored['cell_type_label'].isin(inhib_labels)][SCORE_COL]

# Mann-Whitney U: neuronal vs non-neuronal
U_nn, p_nn = scipy_stats.mannwhitneyu(neuro_scores, nonneuro_scores, alternative='two-sided')
rbc_nn     = (2 * U_nn) / (len(neuro_scores) * len(nonneuro_scores)) - 1

# Mann-Whitney U: excitatory vs inhibitory
U_ei, p_ei = scipy_stats.mannwhitneyu(excit_scores, inhib_scores, alternative='two-sided')
rbc_ei     = (2 * U_ei) / (len(excit_scores) * len(inhib_scores)) - 1

top_type   = subtype_stats['median'].idxmax()
top_med    = subtype_stats.loc[top_type, 'median']

# ── Summary box-and-whisker per group, with significance brackets ─────────
group_data = {
    'Excitatory\n(cortical)': excit_scores,
    'Inhibitory\n(cortical)': inhib_scores,
    'Non-neuronal':            nonneuro_scores,
}
colors_box = ['#e63946', '#5b4fcf', '#9e9e9e']
labels     = list(group_data.keys())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
bp = ax.boxplot(
    [group_data[k].values for k in labels],
    patch_artist=True,
    medianprops=dict(color='white', linewidth=2),
    flierprops=dict(marker='.', markersize=1, alpha=0.15, linestyle='none'),
    whiskerprops=dict(linewidth=1.2),
    capprops=dict(linewidth=1.2),
    boxprops=dict(linewidth=1.2),
    widths=0.5,
)
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.85)
for flier, color in zip(bp['fliers'], colors_box):
    flier.set_markerfacecolor(color)
    flier.set_markeredgecolor(color)

ax.set_xticks(range(1, len(labels) + 1))
ax.set_xticklabels(labels)
ax.set_ylabel('Proteomics score', fontsize=11)
ax.set_title('Score by broad cell class', fontsize=11)

# Pairwise significance brackets (Bonferroni correction × 3 comparisons)
def p_to_stars(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

sig_pairs = [(1, 2, excit_scores, inhib_scores),
             (1, 3, excit_scores, nonneuro_scores),
             (2, 3, inhib_scores, nonneuro_scores)]

# whisker tops as bracket base
whis_tops = [w.get_ydata()[1] for w in bp['whiskers'][1::2]]
y_base = max(whis_tops)
data_range = max(s.max() for s in group_data.values()) - min(s.min() for s in group_data.values())
tick = max(0.05, data_range * 0.06)
ax.set_ylim(ax.get_ylim()[0], y_base + tick * 4.5)

for k, (xi, xj, s1, s2) in enumerate(sig_pairs):
    _, p_raw = scipy_stats.mannwhitneyu(s1, s2, alternative='two-sided')
    p_corr   = min(p_raw * 3, 1.0)   # Bonferroni
    stars    = p_to_stars(p_corr)
    yb       = y_base + tick * (k + 1)
    ax.plot([xi, xi, xj, xj], [yb - tick * 0.15, yb, yb, yb - tick * 0.15],
            color='black', lw=1.2, solid_capstyle='round', clip_on=False)
    ax.text((xi + xj) / 2, yb + tick * 0.05, stars,
            ha='center', va='bottom', fontsize=10.5, fontweight='bold')

# Per-subtype dot plot (median ± SEM)
ax = axes[1]
for j, lbl in enumerate(reversed(VIOLIN_ORDER)):
    if lbl not in subtype_stats.index: continue
    row = subtype_stats.loc[lbl]
    col = LABEL_COLORS.get(lbl, '#aaa')
    ax.errorbar(row['median'], j, xerr=row['std']/np.sqrt(row['n']),
                fmt='o', color=col, ms=6, ecolor=col, elinewidth=1, capsize=3)
    ax.text(row['median'] + 0.002, j, lbl, va='center', fontsize=7.5)
ax.axvline(0, color='black', lw=0.8, ls='--')
ax.set_yticks([])
ax.set_xlabel('Median proteomics score ± SEM', fontsize=10)
ax.set_title('Per-subtype median score', fontsize=11)

plt.suptitle('mPFC Proteomics — Cell Type Enrichment Summary', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'score_enrichment_summary.pdf', dpi=600, bbox_inches='tight')
plt.savefig(OUTPUT_DIR / 'score_enrichment_summary.svg', dpi=600, bbox_inches='tight')
plt.show()

# ── Prediction printout ───────────────────────────────────────────────────
print('=' * 58)
print('  CELL CLASS PREDICTION')
print('=' * 58)
print(f'  Neuronal  median = {neuro_scores.median():.4f}  (n={len(neuro_scores):,})')
print(f'  Non-neuro median = {nonneuro_scores.median():.4f}  (n={len(nonneuro_scores):,})')
print(f'  Mann-Whitney (neuro vs non-neuro): p = {p_nn:.2e}, rbc = {rbc_nn:+.3f}')
print()
print(f'  Excitatory median = {excit_scores.median():.4f}  (n={len(excit_scores):,})')
print(f'  Inhibitory median = {inhib_scores.median():.4f}  (n={len(inhib_scores):,})')
print(f'  Mann-Whitney (excit vs inhib):     p = {p_ei:.2e}, rbc = {rbc_ei:+.3f}')
print()
print(f'  Top-scoring subtype : {top_type}  (median = {top_med:.4f})')
print()
if p_nn < 0.05 and rbc_nn > 0:
    verdict = 'NEURONAL'
    if p_ei < 0.05 and rbc_ei > 0:
        verdict += ' — EXCITATORY'
    elif p_ei < 0.05:
        verdict += ' — INHIBITORY'
    else:
        verdict += ' — mixed excitatory/inhibitory'
elif p_nn < 0.05 and rbc_nn < 0:
    verdict = 'NON-NEURONAL'
else:
    verdict = 'AMBIGUOUS (no significant enrichment in either class)'
print(f'  Predicted origin    : {verdict}')
print('=' * 58)


In [ ]:
subtype_stats.to_csv(OUTPUT_DIR / 'score_by_subtype.csv')
labeled_scored[[SCORE_COL, 'cell_type_label', 'is_neuronal']].to_csv(
    OUTPUT_DIR / 'per_cell_scores.csv'
)
print(f'Saved results to {OUTPUT_DIR}/')


In [ ]:
print("Analysis complete.")


## Reproducibility

```
Expression source : Allen Brain Cell Atlas WMB-10X — Isocortex
                    Yao et al., Nature 2023  https://doi.org/10.1038/s41586-023-06812-z
Metadata source   : Allen Brain Cell Atlas abc_atlas_access, manifest 20230830
Normalization     : log2(CPM/10 + 1) — pre-applied in Allen log2 h5ad files
Score method      : sc.tl.score_genes  (Seurat AddModuleScore)
                    Tirosh et al., Science 2016 / Seurat v2+
Random seed       : 42
Cell type labels  : Allen WMB-taxonomy subclass → custom 24-label mapping
Statistics        : Mann-Whitney U test (scipy.stats), rank-biserial correlation
```

**Interpreting scores:**  
Score > 0 → cell expresses proteomics genes *above* what expression-matched background predicts.  
Rank-biserial correlation (rbc): +1 = neuronal always higher; −1 = non-neuronal always higher; 0 = no difference.  
For a negative control, re-run Section 8 with a size-matched random gene list.
